# Exercise 3.1: Mainz OSM Query Planner with an LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/03_llm_basics/notebooks/exercise_3_1_llm_osm_tool_widget_mainz.ipynb)

This notebook uses a Uni Mainz KI-Chat@JGU model to translate natural-language questions into an OpenStreetMap query plan for Mainz.

The workflow is deliberately split into visible steps:

1. Ask the model for a JSON plan.
2. Display the raw model response.
3. Display the exact `overpass_ql` returned by the model.
4. Validate the plan and, only if necessary, build a safe fallback Overpass query from the model's tags.
5. Execute the Overpass query in a separate cell.
6. Convert the result to a GeoDataFrame and visualize it on a map.

Routing is not included here. OpenRouteService routing is a separate spatial-analysis exercise.

## Learning Outcomes

After this exercise you should be able to:

- Call the OpenAI-compatible Uni Mainz KI-Chat@JGU API.
- Prompt an LLM to produce a structured OSM query plan.
- Inspect the model's raw output and generated Overpass QL.
- Validate or repair small omissions before executing a query.
- Execute Overpass queries against live OSM data.
- Use GeoPandas for simple distance filtering.
- Visualize results with Folium in Colab.

## 1. Colab Setup

In [ ]:
!pip -q install geopandas shapely pyproj folium ipywidgets requests pandas


## 2. Imports and Mainz Constants

Use `EPSG:4326` for APIs and maps. Use `EPSG:25832` for metric distance calculations around Mainz.

In [ ]:
from __future__ import annotations

import copy
import getpass
import json
import re
import time
from typing import Any

import geopandas as gpd
import pandas as pd
import requests
from IPython.display import Markdown, display
from shapely.geometry import Point

import folium
from folium.plugins import MarkerCluster

WGS84 = "EPSG:4326"
METRIC_CRS = "EPSG:25832"

# Overpass bbox order: south, west, north, east.
MAINZ_BBOX = {"south": 49.90, "west": 8.13, "north": 50.05, "east": 8.36}
MAINZ_CENTER = (49.9929, 8.2473)  # lat, lon

KNOWN_PLACES = {
    "mainz_hbf": {"label": "Mainz Hauptbahnhof", "lat": 50.0010, "lon": 8.2587},
    "jgu": {"label": "Johannes Gutenberg University Mainz", "lat": 49.9936, "lon": 8.2419},
    "mainz_dom": {"label": "Mainz Cathedral", "lat": 49.9995, "lon": 8.2742},
    "mainz_theater": {"label": "Staatstheater Mainz", "lat": 50.0002, "lon": 8.2714},
}

ALLOWED_TOOLS = {"overpass_search", "osm_buffer_search", "osm_nearest"}
ALLOWED_OSM_KEYS = {"amenity", "tourism", "shop", "leisure", "public_transport", "railway", "highway"}

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
KI_CHAT_API_BASE_URL = "https://ki-chat.uni-mainz.de/api"

print("Mainz OSM query exercise loaded.")
print("Known places:", ", ".join(KNOWN_PLACES))


## 3. Uni Mainz KI-Chat API

Create an API key in the KI-Chat@JGU web interface and paste it below. Do not store API keys in notebook cells.

Official documentation: https://www.zdv.uni-mainz.de/ki-chat-api-nutzung/

In [ ]:
KI_CHAT_API_KEY = getpass.getpass("Paste your KI-Chat@JGU API key for this session: ").strip()
if not KI_CHAT_API_KEY:
    raise RuntimeError("No API key entered.")

ki_chat_headers = {
    "Authorization": f"Bearer {KI_CHAT_API_KEY}",
    "Content-Type": "application/json",
}
print("KI-Chat API key loaded for this notebook session.")


In [ ]:
def ki_chat_request(method: str, endpoint: str, **kwargs: Any) -> Any:
    url = f"{KI_CHAT_API_BASE_URL}{endpoint}"
    response = requests.request(method, url, headers=ki_chat_headers, timeout=90, **kwargs)
    if not response.ok:
        print(f"HTTP {response.status_code} for {method} {endpoint}")
        try:
            print(json.dumps(response.json(), indent=2, ensure_ascii=False))
        except ValueError:
            print(response.text[:1000])
        response.raise_for_status()
    return response.json()


def preview_json(data: Any, max_chars: int = 4000) -> None:
    text = json.dumps(data, indent=2, ensure_ascii=False)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))


In [ ]:
models_response = ki_chat_request("GET", "/models")
model_ids = [m.get("id") for m in models_response.get("data", []) if m.get("id")]
print("Available model IDs:")
for model_id in model_ids:
    print("-", model_id)

PREFERRED_CHAT_MODELS = [
    "Qwen3 235B",
    "Qwen3 235B VL",
    "GPT OSS 120B",
]

chat_model = next((model for model in PREFERRED_CHAT_MODELS if model in model_ids), None)
if chat_model is None:
    chat_model = model_ids[0] if model_ids else PREFERRED_CHAT_MODELS[0]

print(f"\nUsing chat model: {chat_model}")
print("If the model returns no JSON or poor Overpass QL, set chat_model manually to another listed chat/instruct model.")


## 4. Prompt Contract

The LLM should return one JSON object. It may include `overpass_ql`; if it does not, Python will show that omission and build a safe fallback query from `osm_tags`, `center`, and `radius_m`.

Supported tools:

- `overpass_search`: query OSM features in the Mainz bounding box.
- `osm_buffer_search`: query OSM features around a known Mainz point and filter by radius.
- `osm_nearest`: query OSM features around a known Mainz point and return the nearest features.

Expected JSON fields for OSM tools:

- `tool`
- `question`
- `osm_tags`
- `center`, for radius or nearest tasks
- `radius_m`, for radius tasks
- `limit`, for nearest tasks
- `overpass_ql`, optional but useful for debugging
- `analysis`
- `map_title`

In [ ]:
TOOL_PLANNER_SYSTEM_PROMPT = """
You are a geospatial planning assistant for a teaching notebook.
Return exactly one JSON object and no Markdown.

Scope:
- The study area is Mainz, Germany only.
- Use this Mainz bbox in Overpass queries: south={south}, west={west}, north={north}, east={east}.
- Never create Germany-wide, Europe-wide, or global queries.
- Maximum radius_m is 5000.
- Prefer known place IDs: {place_ids}.

Allowed tools: overpass_search, osm_buffer_search, osm_nearest.
Allowed OSM keys: amenity, tourism, shop, leisure, public_transport, railway, highway.
For bicycle parking, use osm_tags={{"amenity": "bicycle_parking"}}.

For OSM tools include: tool, question, osm_tags, analysis, map_title.
For radius or nearest tasks also include: center and radius_m or limit.
Include overpass_ql if you can. If you include it, it must be valid Overpass QL.

Overpass QL requirements:
- Include [out:json][timeout:25];
- Use node/way/relation clauses with the Mainz bbox or an around radius.
- End with out center tags;
""".format(
    south=MAINZ_BBOX["south"],
    west=MAINZ_BBOX["west"],
    north=MAINZ_BBOX["north"],
    east=MAINZ_BBOX["east"],
    place_ids=", ".join(KNOWN_PLACES.keys()),
).strip()


def response_text_from_chat_response(response: dict[str, Any]) -> str:
    choices = response.get("choices") or []
    if not choices:
        raise RuntimeError("The chat response did not contain any choices.")
    choice = choices[0]
    message = choice.get("message") or {}
    content = message.get("content")

    if isinstance(content, str) and content.strip():
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(item))
        text = "".join(parts).strip()
        if text:
            return text
    for key in ["reasoning_content", "reasoning", "text"]:
        value = message.get(key) or choice.get(key)
        if isinstance(value, str) and value.strip():
            return value

    preview_json(response, max_chars=2500)
    raise RuntimeError("The chat response did not contain text. Choose another listed chat_model.")


def extract_json_object(text: str | None) -> dict[str, Any]:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Expected non-empty text containing one JSON object.")
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            print(text[:2000])
            raise ValueError("The model response did not contain a parseable JSON object.")
        return json.loads(match.group(0))


def call_tool_planner(question: str, model: str | None = None) -> dict[str, Any]:
    payload = {
        "model": model or chat_model,
        "messages": [
            {"role": "system", "content": TOOL_PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        "temperature": 0.1,
        "max_tokens": 1500,
    }
    return ki_chat_request("POST", "/chat/completions", json=payload)


def parse_tool_planner_response(response: dict[str, Any]) -> tuple[str, dict[str, Any]]:
    raw_text = response_text_from_chat_response(response)
    plan = extract_json_object(raw_text)
    return raw_text, plan


## 5. Validation and Query Construction

The validator checks the model plan. If the model did not include `overpass_ql`, `normalize_tool_plan()` builds a safe Mainz-scoped query and stores it as `overpass_ql_final`. The original model query stays visible as `overpass_ql_model`.

In [ ]:
class PlanValidationError(ValueError):
    pass


def point_is_inside_mainz(lat: float, lon: float, margin: float = 0.03) -> bool:
    return (
        MAINZ_BBOX["south"] - margin <= lat <= MAINZ_BBOX["north"] + margin
        and MAINZ_BBOX["west"] - margin <= lon <= MAINZ_BBOX["east"] + margin
    )


def validate_osm_tags(osm_tags: dict[str, Any]) -> None:
    if not isinstance(osm_tags, dict) or not osm_tags:
        raise PlanValidationError("osm_tags must be a non-empty dictionary.")
    for key, value in osm_tags.items():
        if key not in ALLOWED_OSM_KEYS:
            raise PlanValidationError(f"OSM key {key!r} is not allowed.")
        if value is not True and not isinstance(value, str):
            raise PlanValidationError(f"OSM value for {key!r} must be a string or true.")


def validate_center(center: dict[str, Any]) -> None:
    if not isinstance(center, dict):
        raise PlanValidationError("center must be a dictionary.")
    lat = float(center.get("lat"))
    lon = float(center.get("lon"))
    if not point_is_inside_mainz(lat, lon):
        raise PlanValidationError(f"Center point is outside Mainz: {(lat, lon)}")


def osm_tag_filter(osm_tags: dict[str, Any]) -> str:
    filters = []
    for key, value in osm_tags.items():
        filters.append(f'["{key}"]' if value is True else f'["{key}"="{value}"]')
    return "".join(filters)


def build_overpass_query_from_plan(plan: dict[str, Any]) -> str:
    tag_filter = osm_tag_filter(plan.get("osm_tags", {}))
    tool = plan.get("tool")

    if tool in {"osm_buffer_search", "osm_nearest"} and isinstance(plan.get("center"), dict):
        center = plan["center"]
        radius_m = int(plan.get("radius_m") or plan.get("search_radius_m") or 1500)
        radius_m = max(1, min(radius_m, 5000))
        plan.setdefault("radius_m", radius_m)
        spatial_filter = f"(around:{radius_m},{float(center['lat'])},{float(center['lon'])})"
    else:
        spatial_filter = f"({MAINZ_BBOX['south']},{MAINZ_BBOX['west']},{MAINZ_BBOX['north']},{MAINZ_BBOX['east']})"

    return f"""
[out:json][timeout:25];
(
  node{tag_filter}{spatial_filter};
  way{tag_filter}{spatial_filter};
  relation{tag_filter}{spatial_filter};
);
out center tags;
""".strip()


def validate_overpass_ql(overpass_ql: str) -> None:
    if not isinstance(overpass_ql, str) or not overpass_ql.strip():
        raise PlanValidationError("overpass_ql must be a non-empty string.")
    ql = overpass_ql.lower()
    if "[out:json" not in ql:
        raise PlanValidationError("Overpass QL must request JSON output.")
    if "out" not in ql:
        raise PlanValidationError("Overpass QL must include an out statement.")
    if "{{bbox}}" in ql:
        raise PlanValidationError("Do not use the dynamic {{bbox}} placeholder.")
    has_bbox = all(str(value)[:4] in overpass_ql for value in MAINZ_BBOX.values())
    has_around = "around:" in ql
    if not has_bbox and not has_around:
        raise PlanValidationError("Overpass QL must use the Mainz bbox or an around radius.")
    if any(word in ql for word in ["germany", "deutschland", "europe"]):
        raise PlanValidationError("The query appears too broad for this Mainz exercise.")


def normalize_tool_plan(plan: dict[str, Any]) -> dict[str, Any]:
    normalized = copy.deepcopy(plan)
    if not isinstance(normalized, dict):
        return normalized

    osm_tags = normalized.get("osm_tags")
    if isinstance(osm_tags, dict) and "bicycle_parking" in osm_tags and "amenity" not in osm_tags:
        osm_tags["amenity"] = "bicycle_parking"
        osm_tags.pop("bicycle_parking", None)

    model_query = normalized.get("overpass_ql")
    normalized["overpass_ql_model"] = model_query if isinstance(model_query, str) and model_query.strip() else None

    if normalized.get("tool") in ALLOWED_TOOLS:
        if normalized["overpass_ql_model"]:
            normalized["overpass_ql_final"] = normalized["overpass_ql_model"]
        else:
            normalized["overpass_ql_final"] = build_overpass_query_from_plan(normalized)
        normalized["overpass_ql"] = normalized["overpass_ql_final"]

    return normalized


def validate_tool_plan(plan: dict[str, Any]) -> dict[str, Any]:
    if not isinstance(plan, dict):
        raise PlanValidationError("Tool plan must be a JSON object.")

    normalized = normalize_tool_plan(plan)
    tool = normalized.get("tool")
    if tool not in ALLOWED_TOOLS:
        raise PlanValidationError(f"Tool {tool!r} is not allowed.")
    if not isinstance(normalized.get("question"), str) or not normalized["question"].strip():
        raise PlanValidationError("Plan must include the original question.")

    validate_osm_tags(normalized.get("osm_tags", {}))
    if tool in {"osm_buffer_search", "osm_nearest"}:
        validate_center(normalized.get("center", {}))
    if tool == "osm_buffer_search":
        radius_m = int(normalized.get("radius_m", 0))
        if radius_m <= 0 or radius_m > 5000:
            raise PlanValidationError("radius_m must be between 1 and 5000.")
    if tool == "osm_nearest":
        limit = int(normalized.get("limit", 5))
        if limit <= 0 or limit > 20:
            raise PlanValidationError("limit must be between 1 and 20.")

    validate_overpass_ql(normalized.get("overpass_ql_final", ""))
    return normalized


def show_overpass_queries(plan: dict[str, Any]) -> None:
    model_query = plan.get("overpass_ql_model")
    final_query = plan.get("overpass_ql_final") or plan.get("overpass_ql")
    print("MODEL RETURNED overpass_ql:")
    print(model_query if model_query else "<model did not return overpass_ql>")
    print("\nQUERY THAT WILL BE EXECUTED:")
    print(final_query)


## 6. Ask the Model and Inspect Its Raw Output

This cell does not validate or execute anything. It only shows what the model returned.

In [ ]:
question = "Find cafes within 800 meters of Mainz Hauptbahnhof."
planner_response = call_tool_planner(question)
raw_model_text, model_plan = parse_tool_planner_response(planner_response)

print("RAW MODEL TEXT:")
print(raw_model_text)
print("\nPARSED JSON PLAN:")
preview_json(model_plan)
print("\nMODEL RETURNED overpass_ql:")
print(model_plan.get("overpass_ql") or "<model did not return overpass_ql>")


## 7. Validate the Plan and Choose the Query

If the model did not return `overpass_ql`, this cell builds a safe fallback query. It still shows both the model query and the final query.

In [ ]:
validated_plan = validate_tool_plan(model_plan)
preview_json(validated_plan)
print()
show_overpass_queries(validated_plan)


## 8. Execute the Overpass Query

Now execute only the query shown above. If this fails, debug the Overpass QL first before continuing.

In [ ]:
def run_overpass_query(overpass_ql: str, pause_seconds: float = 1.0) -> dict[str, Any]:
    time.sleep(pause_seconds)
    response = requests.post(OVERPASS_URL, data={"data": overpass_ql}, timeout=60)
    if not response.ok:
        print(response.text[:1500])
        response.raise_for_status()
    return response.json()


overpass_json = run_overpass_query(validated_plan["overpass_ql_final"])
elements = overpass_json.get("elements", [])
print(f"Overpass returned {len(elements)} raw elements.")
print("First raw element, if available:")
preview_json(elements[0] if elements else {})


## 9. Convert OSM Elements to a GeoDataFrame

In [ ]:
def osm_elements_to_gdf(overpass_json: dict[str, Any]) -> gpd.GeoDataFrame:
    rows = []
    for element in overpass_json.get("elements", []):
        tags = element.get("tags", {}) or {}
        lat = element.get("lat")
        lon = element.get("lon")
        if lat is None or lon is None:
            center = element.get("center") or {}
            lat = center.get("lat")
            lon = center.get("lon")
        if lat is None or lon is None:
            continue
        row = {
            "osm_type": element.get("type"),
            "osm_id": element.get("id"),
            "name": tags.get("name"),
            "geometry": Point(float(lon), float(lat)),
        }
        row.update(tags)
        rows.append(row)
    if not rows:
        return gpd.GeoDataFrame(columns=["osm_type", "osm_id", "name", "geometry"], geometry="geometry", crs=WGS84)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=WGS84)


def add_distance_to_center(gdf: gpd.GeoDataFrame, center: dict[str, Any]) -> gpd.GeoDataFrame:
    out = gdf.copy()
    if out.empty:
        out["distance_m"] = pd.Series(dtype="float64")
        return out
    center_gdf = gpd.GeoDataFrame(geometry=[Point(float(center["lon"]), float(center["lat"]))], crs=WGS84).to_crs(METRIC_CRS)
    metric = out.to_crs(METRIC_CRS)
    out["distance_m"] = metric.distance(center_gdf.geometry.iloc[0]).round(1).values
    return out


def apply_plan_analysis(gdf: gpd.GeoDataFrame, plan: dict[str, Any]) -> gpd.GeoDataFrame:
    tool = plan["tool"]
    if tool == "osm_buffer_search":
        out = add_distance_to_center(gdf, plan["center"])
        return out[out["distance_m"] <= int(plan["radius_m"])].sort_values("distance_m").reset_index(drop=True)
    if tool == "osm_nearest":
        out = add_distance_to_center(gdf, plan["center"])
        return out.sort_values("distance_m").head(int(plan.get("limit", 5))).reset_index(drop=True)
    return gdf


def show_table(gdf: gpd.GeoDataFrame, n: int = 10) -> None:
    if gdf.empty:
        display(Markdown("No features returned."))
        return
    preferred = ["name", "amenity", "tourism", "shop", "highway", "distance_m"]
    columns = [col for col in preferred if col in gdf.columns]
    display(gdf[columns].head(n) if columns else gdf.drop(columns="geometry").head(n))


raw_gdf = osm_elements_to_gdf(overpass_json)
result_gdf = apply_plan_analysis(raw_gdf, validated_plan)
print(f"GeoDataFrame rows after analysis: {len(result_gdf)}")
show_table(result_gdf)


## 10. Map Visualization

In [ ]:
def make_base_map(title: str | None = None) -> folium.Map:
    m = folium.Map(location=MAINZ_CENTER, zoom_start=13, tiles="OpenStreetMap", control_scale=True)
    if title:
        html = f'<div style="position: fixed; top: 10px; left: 50px; z-index: 9999; background: white; padding: 8px 10px; border: 1px solid #999; font-size: 14px;"><strong>{title}</strong></div>'
        m.get_root().html.add_child(folium.Element(html))
    return m


def add_center_to_map(m: folium.Map, center: dict[str, Any], radius_m: int | None = None) -> folium.Map:
    lat, lon = float(center["lat"]), float(center["lon"])
    folium.Marker((lat, lon), popup=center.get("label", "Center"), icon=folium.Icon(color="red")).add_to(m)
    if radius_m:
        folium.Circle((lat, lon), radius=radius_m, color="red", fill=False).add_to(m)
    return m


def add_gdf_to_map(m: folium.Map, gdf: gpd.GeoDataFrame) -> folium.Map:
    if gdf.empty:
        return m
    cluster = MarkerCluster(name="OSM features").add_to(m)
    for _, row in gdf.iterrows():
        lon, lat = row.geometry.x, row.geometry.y
        parts = []
        if pd.notna(row.get("name")):
            parts.append(str(row.get("name")))
        for key in ["amenity", "tourism", "shop", "leisure", "highway"]:
            if key in row and pd.notna(row.get(key)):
                parts.append(f"{key}={row.get(key)}")
        if "distance_m" in row and pd.notna(row.get("distance_m")):
            parts.append(f"distance={row.get('distance_m'):.0f} m")
        popup = "<br>".join(parts) if parts else f"OSM {row.get('osm_type')} {row.get('osm_id')}"
        folium.CircleMarker((lat, lon), radius=5, color="blue", fill=True, fill_opacity=0.75, popup=popup).add_to(cluster)
    return m


m = make_base_map(validated_plan.get("map_title", "Mainz OSM result"))
if "center" in validated_plan:
    add_center_to_map(m, validated_plan["center"], validated_plan.get("radius_m"))
add_gdf_to_map(m, result_gdf)
folium.LayerControl().add_to(m)
display(m)


## 11. Repeatable Test Prompts

Run the cells above again with these questions. For each test, inspect the model query before executing it.

1. `Find cafes within 800 meters of Mainz Hauptbahnhof.`
2. `Show the five nearest bicycle parking locations to Johannes Gutenberg University Mainz.`
3. `Find museums in central Mainz.`
4. `Find supermarkets near Mainz Cathedral.`
5. Guardrail test: `Find all restaurants in Germany.` The plan should be narrowed to Mainz or rejected before execution.

## 12. Student Tasks

1. Compare the model's `overpass_ql` with the final executed query.
2. Edit a broken model-generated Overpass query manually and test it again.
3. Add one more allowed OSM tag key and explain why it is safe for this exercise.
4. Add one validation rule that rejects overly broad queries.
5. Write a short note explaining why visible intermediate steps are useful in LLM tool-use workflows.